# CarDD Annotation Conversion

## 1. Objective

This notebook converts the CarDD (Car Damage Detection) dataset annotations from COCO
format into the Ultralytics YOLO instance-segmentation format.

CarDD is distributed as COCO JSON: one annotation file per split, containing an `images`
list, a `categories` list, and an `annotations` list whose `segmentation` polygons are
expressed in absolute pixel coordinates. Ultralytics cannot train a segmentation model
directly from that representation. It expects one plain-text label file per image, with
one line per polygon, a zero-based class index, and polygon coordinates normalised to the
range 0-1 by the image width and height.

The Carparts-Seg dataset inspected in `03_carparts_preprocessing.ipynb` is already stored
in that YOLO layout. Converting CarDD to the same layout gives both datasets a single
common format, so the damage model and the vehicle-part model can be trained and evaluated
through one pipeline.

The notebook performs the conversion, prepares the YOLO directory structure, generates the
Ultralytics dataset configuration file, validates the converted labels, and reports a
summary of the results.

**Execution note.** The CarDD dataset is not included in this repository. Sections 4 to 12
require the raw dataset to be present locally. Every one of those sections is guarded and
will print instructions instead of raising an error when the dataset is missing.

## 2. Imports and Configuration

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import json
import os
import shutil
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon
from PIL import Image
import yaml

In [ ]:
# Fixed seed so that the visual validation samples are reproducible between runs.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Splits handled by this notebook, in the order used throughout the report.
SPLITS = ["train", "val", "test"]

# Minimum number of points required for a polygon to describe an area.
MIN_POLYGON_POINTS = 3

# Number of decimal places written for each normalised coordinate.
COORDINATE_PRECISION = 6

## 3. Dataset Path Configuration

Paths are declared relative to the notebook and then resolved to absolute paths, so the
notebook behaves identically on any operating system and on any machine. No user-specific
or drive-specific path is hard-coded.

In [ ]:
RAW_ROOT = Path("../data/cardd/raw").resolve()
PROCESSED_ROOT = Path("../data/cardd/processed").resolve()

print("Raw dataset root:      ", RAW_ROOT)
print("Processed output root: ", PROCESSED_ROOT)
print("Raw dataset exists:    ", RAW_ROOT.exists())

### 3.1 Locating the COCO Annotation Files

CarDD is published with COCO-style split names (`train2017`, `val2017`, `test2017`), but
different mirrors and manual re-packagings use slightly different folder depths and file
names. Rather than hard-coding one layout, the annotation file for each split is discovered
by searching the raw dataset tree for a JSON file whose name contains the split keyword.

In [ ]:
# Split name -> keywords that may appear in the corresponding annotation file name.
# "val" is listed before "test" only for readability; matching is exact per split.
SPLIT_KEYWORDS = {
    "train": ("train",),
    "val": ("val", "valid"),
    "test": ("test",),
}


def find_annotation_files(raw_root: Path) -> dict:
    """Return {split: Path} for every COCO annotation JSON found under raw_root.

    The search is recursive and case-insensitive. Only files that actually parse as COCO
    (an object with an "images" key) are accepted, so unrelated JSON files in the tree do
    not produce false matches.
    """
    found = {}
    if not raw_root.exists():
        return found

    candidates = sorted(raw_root.rglob("*.json"))

    for split, keywords in SPLIT_KEYWORDS.items():
        for candidate in candidates:
            name = candidate.name.lower()
            if not any(keyword in name for keyword in keywords):
                continue
            # "instances_train2017.json" must not be picked up as the val split, and so on.
            other_keywords = [
                kw for other, kws in SPLIT_KEYWORDS.items() if other != split for kw in kws
            ]
            if any(kw in name for kw in other_keywords if kw not in keywords):
                continue
            try:
                with open(candidate, "r", encoding="utf-8") as file:
                    peek = json.load(file)
            except (json.JSONDecodeError, OSError):
                continue
            if isinstance(peek, dict) and "images" in peek:
                found[split] = candidate
                break

    return found


annotation_files = find_annotation_files(RAW_ROOT)

DATASET_AVAILABLE = len(annotation_files) > 0

for split in SPLITS:
    path = annotation_files.get(split)
    print(f"{split:5s} annotations: {path if path is not None else 'not found'}")

print()
print("Dataset available:", DATASET_AVAILABLE)

In [ ]:
if not DATASET_AVAILABLE:
    print("The CarDD dataset was not found. Sections 4 to 12 will be skipped.")
    print()
    print("To run this notebook, download CarDD and place the COCO release under:")
    print("   ", RAW_ROOT)
    print()
    print("A typical CarDD COCO layout looks like this:")
    print("    data/cardd/raw/")
    print("        annotations/")
    print("            instances_train2017.json")
    print("            instances_val2017.json")
    print("            instances_test2017.json")
    print("        train2017/   <- image files")
    print("        val2017/")
    print("        test2017/")
    print()
    print("The exact folder depth does not matter; annotation files are discovered")
    print("recursively and image folders are resolved from the file names inside them.")
else:
    print("Dataset located. All sections can be executed.")

## 4. Load and Inspect COCO Annotations

**Requires the dataset.** Each annotation file is loaded once and kept in memory for the
remainder of the notebook. The top-level structure is reported so that any deviation from
the expected COCO schema is visible before the conversion starts.

In [ ]:
coco_data = {}

if DATASET_AVAILABLE:
    for split, path in annotation_files.items():
        with open(path, "r", encoding="utf-8") as file:
            coco_data[split] = json.load(file)

    structure_rows = []
    for split in SPLITS:
        if split not in coco_data:
            continue
        data = coco_data[split]
        structure_rows.append({
            "Split": split,
            "Top-level keys": ", ".join(sorted(data.keys())),
            "Images": len(data.get("images", [])),
            "Annotations": len(data.get("annotations", [])),
            "Categories": len(data.get("categories", [])),
        })

    coco_structure = pd.DataFrame(structure_rows)
    display(coco_structure)
else:
    print("Skipped: dataset not available.")

### 4.1 Sample Image and Annotation Records

The first record of each type is printed to confirm the field names this notebook relies
on: `id`, `file_name`, `width` and `height` for images, and `image_id`, `category_id`,
`iscrowd` and `segmentation` for annotations.

In [ ]:
if DATASET_AVAILABLE and coco_data:
    reference_split = next(iter(coco_data))
    reference = coco_data[reference_split]

    print("Reference split:", reference_split)
    print()

    if reference.get("images"):
        print("Sample image record:")
        print(json.dumps(reference["images"][0], indent=2))
    print()

    if reference.get("annotations"):
        sample_annotation = dict(reference["annotations"][0])
        segmentation = sample_annotation.get("segmentation")
        # The segmentation field is long; summarise it rather than printing every value.
        if isinstance(segmentation, list):
            sample_annotation["segmentation"] = (
                f"<list of {len(segmentation)} polygon(s), "
                f"first polygon has {len(segmentation[0]) if segmentation else 0} values>"
            )
        elif isinstance(segmentation, dict):
            sample_annotation["segmentation"] = "<RLE object>"
        print("Sample annotation record:")
        print(json.dumps(sample_annotation, indent=2))
else:
    print("Skipped: dataset not available.")

### 4.2 Missing Image Dimensions

Normalisation depends on the width and height of every image. COCO records normally carry
both fields, but any record missing them is reported here, because the converter has to
fall back to opening the image file to recover the dimensions.

In [ ]:
missing_dimensions = defaultdict(list)

if DATASET_AVAILABLE:
    for split, data in coco_data.items():
        for image_record in data.get("images", []):
            if not image_record.get("width") or not image_record.get("height"):
                missing_dimensions[split].append(image_record.get("file_name"))

    total_missing_dimensions = sum(len(values) for values in missing_dimensions.values())
    print("Image records without width/height:", total_missing_dimensions)
    for split, values in missing_dimensions.items():
        print(f"  {split}: {len(values)}")
else:
    total_missing_dimensions = 0
    print("Skipped: dataset not available.")

## 5. Inspect and Map Damage Categories

**Requires the dataset.** COCO category identifiers are arbitrary integers and usually
start at 1. Ultralytics requires contiguous class indices starting at 0.

The mapping is built by sorting the COCO categories by their original identifier and
assigning YOLO indices in that order. This rule is deterministic and reproducible, which
matters because the resulting class order is written into `cardd.yaml` and is baked into
any model trained afterwards. Changing the order later would silently invalidate trained
weights.

In [ ]:
category_mapping = {}
class_names = []

if DATASET_AVAILABLE and coco_data:
    # Collect categories from every split and confirm they agree.
    category_sets = {}
    for split, data in coco_data.items():
        categories = {
            category["id"]: category["name"] for category in data.get("categories", [])
        }
        category_sets[split] = categories

    reference_categories = category_sets[next(iter(category_sets))]

    inconsistent_splits = [
        split for split, categories in category_sets.items()
        if categories != reference_categories
    ]

    if inconsistent_splits:
        print("WARNING: category definitions differ between splits:", inconsistent_splits)
    else:
        print("Category definitions are identical across all loaded splits.")

    # COCO category id -> contiguous YOLO class id, ordered by original COCO id.
    sorted_category_ids = sorted(reference_categories.keys())
    category_mapping = {
        coco_id: index for index, coco_id in enumerate(sorted_category_ids)
    }
    class_names = [reference_categories[coco_id] for coco_id in sorted_category_ids]

    category_table = pd.DataFrame({
        "COCO category ID": sorted_category_ids,
        "YOLO class ID": [category_mapping[cid] for cid in sorted_category_ids],
        "Class name": class_names,
    })
    display(category_table)
else:
    print("Skipped: dataset not available.")

In [ ]:
if DATASET_AVAILABLE and class_names:
    number_of_classes = len(class_names)
    print("Number of classes:", number_of_classes)
    print("Final class ordering used for cardd.yaml and for all training runs:")
    for index, name in enumerate(class_names):
        print(f"  {index}: {name}")
else:
    number_of_classes = 0
    print("Skipped: dataset not available.")

## 6. COCO-to-YOLO Segmentation Conversion

This section defines the conversion logic. It contains no dataset access, so it runs
whether or not CarDD is present.

### Format

Each YOLO segmentation label file holds one line per polygon:

```text
<class_id> <x1> <y1> <x2> <y2> ... <xn> <yn>
```

All coordinates are normalised: `x` is divided by the image width and `y` by the image
height, giving values in the range 0-1.

### Handling decisions

**Multiple annotations per image.** Every annotation belonging to an image is written into
that image's label file, one line per polygon.

**Multiple polygons per annotation.** COCO stores `segmentation` as a list of polygons, so
a single damage instance split by an occlusion is represented by several polygons. The YOLO
segmentation format has no equivalent grouping: one line is one polygon. Each polygon is
therefore written as a separate line carrying the same class identifier. This preserves the
full annotated shape and the correct class, at the cost of splitting one annotated instance
into several YOLO instances. The alternative approaches were rejected: keeping only the
largest polygon discards annotated damage, and merging the parts with a connecting seam
invents boundary geometry that was never annotated. The number of annotations affected is
counted and reported, so the impact on instance counts is explicit rather than hidden.

**RLE and crowd annotations.** When `segmentation` is a dictionary it is run-length
encoded, not a polygon, and it cannot be converted without a mask-to-contour step. Such
annotations, and any annotation flagged `iscrowd=1`, are skipped and counted rather than
converted incorrectly.

**Invalid annotations.** Polygons with an odd number of coordinate values, with fewer than
three points, with non-finite values, or with zero area are skipped and counted by reason.
Nothing is discarded silently.

**Clipping.** Normalised coordinates are clipped to [0, 1]. CarDD polygons occasionally sit
a fraction of a pixel outside the image bounds; clipping keeps the labels inside the range
Ultralytics accepts. The number of clipped coordinate values is counted, because a large
count would indicate a genuine mismatch between the annotations and the images rather than
harmless rounding.

**Images with no annotations.** An empty label file is created. Ultralytics reads such an
image as a valid background sample. The count is reported, since notebook 03 found that
empty labels in Carparts-Seg often signalled missing annotations rather than true negatives.

In [ ]:
def polygon_to_yolo(polygon, image_width, image_height):
    """Normalise one flat COCO polygon to YOLO coordinates.

    polygon is a flat sequence [x1, y1, x2, y2, ...] in absolute pixels.

    Returns (coordinates, reason, clipped_count):
      coordinates   - flat list of normalised values, or None if the polygon is invalid
      reason        - None when valid, otherwise a short rejection reason
      clipped_count - number of individual values that had to be clipped into [0, 1]
    """
    if len(polygon) % 2 != 0:
        return None, "odd_coordinate_count", 0

    if len(polygon) // 2 < MIN_POLYGON_POINTS:
        return None, "insufficient_points", 0

    try:
        points = np.asarray(polygon, dtype=float).reshape(-1, 2)
    except (ValueError, TypeError):
        return None, "non_numeric", 0

    if not np.isfinite(points).all():
        return None, "non_numeric", 0

    if image_width <= 0 or image_height <= 0:
        return None, "invalid_image_size", 0

    normalised = points / np.array([image_width, image_height], dtype=float)

    clipped_count = int(np.count_nonzero((normalised < 0.0) | (normalised > 1.0)))
    normalised = np.clip(normalised, 0.0, 1.0)

    # Shoelace formula. A zero area means the polygon is degenerate (a point or a line)
    # and carries no segmentation information.
    x = normalised[:, 0]
    y = normalised[:, 1]
    area = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
    if area <= 0.0:
        return None, "zero_area", clipped_count

    return normalised.reshape(-1).tolist(), None, clipped_count


def format_label_line(class_id, coordinates):
    """Render one YOLO segmentation line."""
    values = " ".join(f"{value:.{COORDINATE_PRECISION}f}" for value in coordinates)
    return f"{class_id} {values}"

In [ ]:
def convert_split(coco, split_name, images_output_dir, labels_output_dir,
                  image_source_dirs, category_mapping):
    """Convert one COCO split to YOLO segmentation labels and stage its images.

    Returns a statistics dictionary describing everything that happened, including the
    annotations that were skipped and why.
    """
    images_output_dir.mkdir(parents=True, exist_ok=True)
    labels_output_dir.mkdir(parents=True, exist_ok=True)

    # Group annotations by image so that each label file is written exactly once.
    annotations_by_image = defaultdict(list)
    for annotation in coco.get("annotations", []):
        annotations_by_image[annotation["image_id"]].append(annotation)

    stats = {
        "split": split_name,
        "images_in_json": len(coco.get("images", [])),
        "annotations_in_json": len(coco.get("annotations", [])),
        "label_files_written": 0,
        "polygons_written": 0,
        "empty_label_files": 0,
        "images_staged": 0,
        "images_not_found": [],
        "multi_polygon_annotations": 0,
        "extra_lines_from_multi_polygon": 0,
        "skipped_iscrowd": 0,
        "skipped_rle": 0,
        "skipped_empty_segmentation": 0,
        "skipped_unknown_category": 0,
        "clipped_coordinate_values": 0,
        "skipped_reasons": Counter(),
        "class_counts": Counter(),
        "dimension_fallbacks": 0,
    }

    for image_record in coco.get("images", []):
        file_name = Path(image_record["file_name"]).name
        width = image_record.get("width")
        height = image_record.get("height")

        source_path = resolve_image_path(file_name, image_source_dirs)

        # Recover missing dimensions from the image file itself when necessary.
        if (not width or not height) and source_path is not None:
            try:
                with Image.open(source_path) as image:
                    width, height = image.size
                stats["dimension_fallbacks"] += 1
            except OSError:
                width, height = None, None

        lines = []
        for annotation in annotations_by_image.get(image_record["id"], []):
            if annotation.get("iscrowd", 0) == 1:
                stats["skipped_iscrowd"] += 1
                continue

            segmentation = annotation.get("segmentation")

            if isinstance(segmentation, dict):
                stats["skipped_rle"] += 1
                continue

            if not segmentation:
                stats["skipped_empty_segmentation"] += 1
                continue

            coco_category_id = annotation.get("category_id")
            if coco_category_id not in category_mapping:
                stats["skipped_unknown_category"] += 1
                continue
            class_id = category_mapping[coco_category_id]

            # A COCO segmentation is a list of polygons; a flat list of numbers is
            # tolerated as a single polygon for robustness against non-standard files.
            if segmentation and not isinstance(segmentation[0], (list, tuple)):
                polygons = [segmentation]
            else:
                polygons = list(segmentation)

            accepted_for_annotation = 0
            for polygon in polygons:
                coordinates, reason, clipped = polygon_to_yolo(polygon, width or 0, height or 0)
                stats["clipped_coordinate_values"] += clipped
                if coordinates is None:
                    stats["skipped_reasons"][reason] += 1
                    continue
                lines.append(format_label_line(class_id, coordinates))
                stats["class_counts"][class_id] += 1
                accepted_for_annotation += 1

            if len(polygons) > 1:
                stats["multi_polygon_annotations"] += 1
                stats["extra_lines_from_multi_polygon"] += max(accepted_for_annotation - 1, 0)

        label_path = labels_output_dir / (Path(file_name).stem + ".txt")
        with open(label_path, "w", encoding="utf-8", newline="\n") as file:
            if lines:
                file.write("\n".join(lines) + "\n")

        stats["label_files_written"] += 1
        stats["polygons_written"] += len(lines)
        if not lines:
            stats["empty_label_files"] += 1

        if source_path is None:
            stats["images_not_found"].append(file_name)
        else:
            if stage_image(source_path, images_output_dir / file_name):
                stats["images_staged"] += 1

    return stats

### 6.1 Image Staging Strategy

The raw images are not copied by default. A hard link is attempted first: it costs no extra
disk space, the raw dataset stays untouched, and the linked file behaves like an ordinary
file to every tool that reads it. Hard links fail across filesystems and on some Windows
configurations, so the function falls back to a symbolic link and finally to a real copy.
The strategy actually used is reported, so the result is never ambiguous.

In [ ]:
# Records which staging strategy succeeded, for the summary in section 12.
staging_strategy_counts = Counter()


def stage_image(source_path: Path, destination_path: Path) -> bool:
    """Make source_path available at destination_path.

    Tries hard link, then symbolic link, then a full copy. Returns True on success.
    """
    if destination_path.exists() or destination_path.is_symlink():
        staging_strategy_counts["already_present"] += 1
        return True

    try:
        os.link(source_path, destination_path)
        staging_strategy_counts["hard_link"] += 1
        return True
    except (OSError, NotImplementedError, AttributeError):
        pass

    try:
        # Symbolic links require developer mode or elevation on Windows; this may fail.
        os.symlink(source_path, destination_path)
        staging_strategy_counts["symlink"] += 1
        return True
    except (OSError, NotImplementedError, AttributeError):
        pass

    try:
        shutil.copy2(source_path, destination_path)
        staging_strategy_counts["copy"] += 1
        return True
    except OSError:
        staging_strategy_counts["failed"] += 1
        return False


def resolve_image_path(file_name: str, source_dirs):
    """Find an image by name in the candidate source directories."""
    for directory in source_dirs:
        candidate = directory / file_name
        if candidate.is_file():
            return candidate
    return None

## 7. Process Train / Validation / Test Splits

**Requires the dataset.** Image folders are located per split by taking the file names
listed in the COCO JSON and searching the raw tree for the directories that contain them.
This avoids assuming a specific folder naming convention.

In [ ]:
def find_image_directories(raw_root: Path, coco, max_probe: int = 25):
    """Return the directories under raw_root that hold this split's image files.

    A sample of file names from the COCO record is looked up in an index of the raw tree,
    so the result reflects where the images actually are rather than an assumed folder name.
    """
    image_records = coco.get("images", [])
    if not image_records:
        return []

    # Index every image file in the raw tree once, by file name.
    index = defaultdict(set)
    image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
    for path in raw_root.rglob("*"):
        if path.is_file() and path.suffix.lower() in image_suffixes:
            index[path.name].add(path.parent)

    directories = Counter()
    for image_record in image_records[:max_probe]:
        name = Path(image_record["file_name"]).name
        for parent in index.get(name, ()):
            directories[parent] += 1

    # Most frequently matching directory first.
    return [directory for directory, _ in directories.most_common()]

In [ ]:
conversion_stats = {}

if DATASET_AVAILABLE:
    for split in SPLITS:
        if split not in coco_data:
            print(f"{split}: no annotation file found, skipping.")
            continue

        source_dirs = find_image_directories(RAW_ROOT, coco_data[split])
        print(f"{split}: image source directories ->",
              [str(d) for d in source_dirs] or "none found")

        stats = convert_split(
            coco=coco_data[split],
            split_name=split,
            images_output_dir=PROCESSED_ROOT / "images" / split,
            labels_output_dir=PROCESSED_ROOT / "labels" / split,
            image_source_dirs=source_dirs,
            category_mapping=category_mapping,
        )
        conversion_stats[split] = stats

        print(f"  label files written: {stats['label_files_written']}"
              f"  polygons: {stats['polygons_written']}"
              f"  images staged: {stats['images_staged']}")
        print()
else:
    print("Skipped: dataset not available.")

### 7.1 Per-Split Conversion Report

Every skipped annotation is accounted for here. `Skipped (other)` aggregates the geometric
rejection reasons, which are broken down individually in the following cell.

In [ ]:
if DATASET_AVAILABLE and conversion_stats:
    conversion_report = pd.DataFrame([
        {
            "Split": stats["split"],
            "Images in JSON": stats["images_in_json"],
            "Annotations in JSON": stats["annotations_in_json"],
            "Label files written": stats["label_files_written"],
            "Polygon lines written": stats["polygons_written"],
            "Empty label files": stats["empty_label_files"],
            "Images staged": stats["images_staged"],
            "Images not found": len(stats["images_not_found"]),
            "Skipped (iscrowd)": stats["skipped_iscrowd"],
            "Skipped (RLE)": stats["skipped_rle"],
            "Skipped (empty segmentation)": stats["skipped_empty_segmentation"],
            "Skipped (unknown category)": stats["skipped_unknown_category"],
            "Skipped (other)": sum(stats["skipped_reasons"].values()),
            "Multi-polygon annotations": stats["multi_polygon_annotations"],
            "Extra lines from multi-polygon": stats["extra_lines_from_multi_polygon"],
            "Clipped coordinate values": stats["clipped_coordinate_values"],
        }
        for stats in conversion_stats.values()
    ])
    display(conversion_report.set_index("Split").T)
else:
    print("Skipped: dataset not available.")

In [ ]:
if DATASET_AVAILABLE and conversion_stats:
    reason_rows = []
    for split, stats in conversion_stats.items():
        for reason, count in sorted(stats["skipped_reasons"].items()):
            reason_rows.append({"Split": split, "Reason": reason, "Count": count})

    if reason_rows:
        display(pd.DataFrame(reason_rows))
    else:
        print("No polygons were rejected for geometric reasons.")

    # Any image listed in the JSON but absent from disk breaks the image/label pairing.
    for split, stats in conversion_stats.items():
        missing = stats["images_not_found"]
        if missing:
            print(f"{split}: {len(missing)} image file(s) referenced but not found. Examples:")
            for name in missing[:5]:
                print("   ", name)
else:
    print("Skipped: dataset not available.")

## 8. Dataset Structure Preparation

**Requires the dataset.** The directories are created by the conversion in section 7. This
section confirms that the resulting tree matches the layout Ultralytics expects and that
`images/<split>` and `labels/<split>` hold the same number of files.

In [ ]:
if DATASET_AVAILABLE:
    structure_rows = []
    image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    for split in SPLITS:
        images_dir = PROCESSED_ROOT / "images" / split
        labels_dir = PROCESSED_ROOT / "labels" / split

        image_count = (
            sum(1 for p in images_dir.iterdir()
                if p.is_file() and p.suffix.lower() in image_suffixes)
            if images_dir.exists() else 0
        )
        label_count = len(list(labels_dir.glob("*.txt"))) if labels_dir.exists() else 0

        structure_rows.append({
            "Split": split,
            "images/ exists": images_dir.exists(),
            "labels/ exists": labels_dir.exists(),
            "Image files": image_count,
            "Label files": label_count,
            "Counts match": image_count == label_count,
        })

    structure_table = pd.DataFrame(structure_rows)
    display(structure_table)

    print("Processed dataset root:", PROCESSED_ROOT)
else:
    print("Skipped: dataset not available.")

## 9. Generate Ultralytics Dataset YAML

**Requires the dataset.** `cardd.yaml` is written programmatically into the processed
dataset folder when this cell runs. It is not stored in the repository, because
`data/` is excluded from version control.

The split paths are written relative to `path`, matching the convention used by
`carparts-seg.yaml` in notebook 03. The `names` mapping reproduces the class ordering fixed
in section 5, and must not be reordered afterwards.

In [ ]:
if DATASET_AVAILABLE and class_names:
    dataset_config = {
        "path": str(PROCESSED_ROOT),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {index: name for index, name in enumerate(class_names)},
    }

    YAML_PATH = PROCESSED_ROOT / "cardd.yaml"
    PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

    with open(YAML_PATH, "w", encoding="utf-8") as file:
        yaml.safe_dump(dataset_config, file, sort_keys=False, allow_unicode=True)

    print("Written:", YAML_PATH)
    print()
    print(YAML_PATH.read_text(encoding="utf-8"))
else:
    print("Skipped: dataset not available.")

## 10. Validate Converted Labels

**Requires the dataset.** The validation reads the written label files back from disk rather
than reusing the in-memory conversion results, so it verifies the files that training will
actually consume. The checks mirror those applied to Carparts-Seg in notebook 03, which
makes the two datasets directly comparable.

In [ ]:
validation = {
    "total_images": 0,
    "total_label_files": 0,
    "missing_labels": [],
    "missing_images": [],
    "invalid_class_ids": [],
    "odd_coordinate_counts": [],
    "insufficient_polygon_points": [],
    "non_numeric_coordinates": [],
    "out_of_range_coordinates": [],
    "zero_area_polygons": [],
    "empty_label_files": [],
}
per_class_counts = Counter()

if DATASET_AVAILABLE:
    image_suffixes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    for split in SPLITS:
        images_dir = PROCESSED_ROOT / "images" / split
        labels_dir = PROCESSED_ROOT / "labels" / split
        if not labels_dir.exists():
            continue

        image_stems = {
            p.stem for p in images_dir.iterdir()
            if p.is_file() and p.suffix.lower() in image_suffixes
        } if images_dir.exists() else set()
        label_paths = sorted(labels_dir.glob("*.txt"))
        label_stems = {p.stem for p in label_paths}

        validation["total_images"] += len(image_stems)
        validation["total_label_files"] += len(label_paths)
        validation["missing_labels"] += [
            f"{split}/{stem}" for stem in sorted(image_stems - label_stems)
        ]
        validation["missing_images"] += [
            f"{split}/{stem}" for stem in sorted(label_stems - image_stems)
        ]

        for label_path in label_paths:
            lines = [
                line.strip()
                for line in label_path.read_text(encoding="utf-8").splitlines()
                if line.strip()
            ]
            if not lines:
                validation["empty_label_files"].append(f"{split}/{label_path.name}")
                continue

            for line_number, line in enumerate(lines, start=1):
                location = f"{split}/{label_path.name}:{line_number}"
                values = line.split()

                try:
                    class_id = int(values[0])
                    coordinates = np.asarray(values[1:], dtype=float)
                except ValueError:
                    validation["non_numeric_coordinates"].append(location)
                    continue

                if not (0 <= class_id < max(number_of_classes, 1)):
                    validation["invalid_class_ids"].append(location)
                    continue

                per_class_counts[class_id] += 1

                if len(coordinates) % 2 != 0:
                    validation["odd_coordinate_counts"].append(location)
                    continue

                points = coordinates.reshape(-1, 2)

                if len(points) < MIN_POLYGON_POINTS:
                    validation["insufficient_polygon_points"].append(location)
                    continue

                if not np.isfinite(points).all():
                    validation["non_numeric_coordinates"].append(location)
                    continue

                if points.min() < 0.0 or points.max() > 1.0:
                    validation["out_of_range_coordinates"].append(location)

                x, y = points[:, 0], points[:, 1]
                area = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
                if area <= 0.0:
                    validation["zero_area_polygons"].append(location)

    print("Validation complete.")
else:
    print("Skipped: dataset not available.")

In [ ]:
if DATASET_AVAILABLE:
    validation_table = pd.DataFrame({
        "Check": [
            "Total images",
            "Total label files",
            "Missing label files",
            "Missing image files",
            "Invalid class IDs",
            "Odd coordinate counts",
            "Polygons with insufficient points",
            "Non-numeric coordinates",
            "Out-of-range coordinates",
            "Zero-area polygons",
            "Empty label files",
        ],
        "Result": [
            validation["total_images"],
            validation["total_label_files"],
            len(validation["missing_labels"]),
            len(validation["missing_images"]),
            len(validation["invalid_class_ids"]),
            len(validation["odd_coordinate_counts"]),
            len(validation["insufficient_polygon_points"]),
            len(validation["non_numeric_coordinates"]),
            len(validation["out_of_range_coordinates"]),
            len(validation["zero_area_polygons"]),
            len(validation["empty_label_files"]),
        ],
    })
    display(validation_table)
else:
    print("Skipped: dataset not available.")

### 10.1 Per-Class Annotation Counts

Counted from the written label files. Because multi-polygon annotations become several
lines, these counts are polygon counts and may exceed the COCO annotation counts; the
difference is reported as `Extra lines from multi-polygon` in section 7.

In [ ]:
if DATASET_AVAILABLE and class_names:
    class_distribution = pd.DataFrame({
        "YOLO class ID": range(number_of_classes),
        "Class name": class_names,
        "Polygon count": [per_class_counts.get(index, 0) for index in range(number_of_classes)],
    }).sort_values("Polygon count", ascending=False)

    display(class_distribution)

    figure, axis = plt.subplots(figsize=(9, 4))
    axis.bar(class_distribution["Class name"], class_distribution["Polygon count"])
    axis.set_title("CarDD polygon count per damage class (converted labels)")
    axis.set_xlabel("Damage class")
    axis.set_ylabel("Polygon count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: dataset not available.")

## 11. Visual Validation of Annotations

**Requires the dataset.** Structural checks cannot detect a systematic error such as
swapped width and height or transposed x and y, which would still produce well-formed
labels. Drawing the converted polygons back onto the images is the only way to confirm the
geometry is correct, so this step should be reviewed before any training run.

In [ ]:
def draw_converted_sample(image_path: Path, label_path: Path, axis, class_names):
    """Overlay the polygons of one converted label file on its image."""
    with Image.open(image_path) as image:
        image = image.convert("RGB")
        width, height = image.size
        axis.imshow(image)

    colours = plt.get_cmap("tab10")

    lines = [
        line.strip()
        for line in label_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]

    for line in lines:
        values = line.split()
        class_id = int(values[0])
        # Scale the normalised coordinates back to pixels for display.
        points = np.asarray(values[1:], dtype=float).reshape(-1, 2)
        points = points * np.array([width, height])

        colour = colours(class_id % 10)
        axis.add_patch(Polygon(points, closed=True, fill=True,
                               facecolor=colour, alpha=0.30))
        axis.add_patch(Polygon(points, closed=True, fill=False,
                               edgecolor=colour, linewidth=1.5))

        label = class_names[class_id] if class_id < len(class_names) else str(class_id)
        axis.text(points[:, 0].min(), points[:, 1].min() - 4, label,
                  fontsize=8, color="white",
                  bbox=dict(facecolor=colour, edgecolor="none", pad=1.5))

    axis.set_title(f"{image_path.name}  ({len(lines)} polygon(s))", fontsize=9)
    axis.axis("off")

In [ ]:
SAMPLE_COUNT = 6
SAMPLE_SPLIT = "train"

if DATASET_AVAILABLE:
    images_dir = PROCESSED_ROOT / "images" / SAMPLE_SPLIT
    labels_dir = PROCESSED_ROOT / "labels" / SAMPLE_SPLIT

    # Only sample images whose label file actually contains polygons.
    annotated_pairs = []
    if images_dir.exists() and labels_dir.exists():
        for label_path in sorted(labels_dir.glob("*.txt")):
            if label_path.stat().st_size == 0:
                continue
            matches = list(images_dir.glob(label_path.stem + ".*"))
            if matches:
                annotated_pairs.append((matches[0], label_path))

    if not annotated_pairs:
        print(f"No annotated samples available in the '{SAMPLE_SPLIT}' split.")
    else:
        selection = random.sample(annotated_pairs, min(SAMPLE_COUNT, len(annotated_pairs)))

        columns = 3
        rows = (len(selection) + columns - 1) // columns
        figure, axes = plt.subplots(rows, columns, figsize=(5 * columns, 4 * rows))
        axes = np.atleast_1d(axes).ravel()

        for axis, (image_path, label_path) in zip(axes, selection):
            draw_converted_sample(image_path, label_path, axis, class_names)

        # Hide any unused subplot slots.
        for axis in axes[len(selection):]:
            axis.axis("off")

        plt.tight_layout()
        plt.show()
else:
    print("Skipped: dataset not available.")

## 12. Conversion Summary

**Requires the dataset.** All figures below are computed at run time from the converted
output. No result is recorded in this notebook in advance.

In [ ]:
if DATASET_AVAILABLE and conversion_stats:
    total_images_in_json = sum(s["images_in_json"] for s in conversion_stats.values())
    total_annotations = sum(s["annotations_in_json"] for s in conversion_stats.values())
    total_polygons = sum(s["polygons_written"] for s in conversion_stats.values())
    total_skipped = sum(
        s["skipped_iscrowd"] + s["skipped_rle"] + s["skipped_empty_segmentation"]
        + s["skipped_unknown_category"] + sum(s["skipped_reasons"].values())
        for s in conversion_stats.values()
    )

    conversion_summary = pd.DataFrame({
        "Item": [
            "Splits converted",
            "Images listed in COCO JSON",
            "Annotations in COCO JSON",
            "Polygon lines written",
            "Label files written",
            "Empty label files",
            "Images staged into processed/",
            "Images referenced but not found",
            "Annotations skipped (all reasons)",
            "Multi-polygon annotations",
            "Extra lines from multi-polygon",
            "Clipped coordinate values",
            "Image records missing width/height",
            "Configured classes",
            "Validation issues (all checks)",
        ],
        "Result": [
            len(conversion_stats),
            total_images_in_json,
            total_annotations,
            total_polygons,
            sum(s["label_files_written"] for s in conversion_stats.values()),
            sum(s["empty_label_files"] for s in conversion_stats.values()),
            sum(s["images_staged"] for s in conversion_stats.values()),
            sum(len(s["images_not_found"]) for s in conversion_stats.values()),
            total_skipped,
            sum(s["multi_polygon_annotations"] for s in conversion_stats.values()),
            sum(s["extra_lines_from_multi_polygon"] for s in conversion_stats.values()),
            sum(s["clipped_coordinate_values"] for s in conversion_stats.values()),
            total_missing_dimensions,
            number_of_classes,
            sum(len(validation[key]) for key in [
                "missing_labels", "missing_images", "invalid_class_ids",
                "odd_coordinate_counts", "insufficient_polygon_points",
                "non_numeric_coordinates", "out_of_range_coordinates",
                "zero_area_polygons",
            ]),
        ],
    })
    display(conversion_summary)

    print("Image staging strategies used:", dict(staging_strategy_counts))
else:
    print("Skipped: dataset not available.")

## 13. Key Findings and Readiness Assessment

### Status

This notebook has not yet been executed against the CarDD dataset, because the dataset is
not present in this working copy. No conversion figures, validation results, or plots are
recorded here. The findings below are the conclusions to be filled in once the notebook has
been run end to end on the real data.

### To record after execution

1. The number of images and annotations in each split, and whether the split sizes match
   the sizes published for CarDD.
2. The final damage-class ordering, copied from section 5, together with a note that this
   ordering is now fixed for every model trained on this converted dataset.
3. The number of annotations skipped, broken down by reason, and whether any RLE or crowd
   annotations were present.
4. The number of multi-polygon annotations, and therefore the gap between the COCO
   annotation count and the YOLO polygon-line count.
5. The number of clipped coordinate values; a large number would indicate a real mismatch
   between the annotations and the images rather than rounding at the image border.
6. The number of empty label files, with a judgement on whether they are true background
   images or missing annotations, following the same question raised for Carparts-Seg in
   notebook 03.
7. The result of the visual validation in section 11, confirming that the polygons align
   with the visible damage.

### Readiness assessment

The converted dataset can be considered ready for Ultralytics segmentation training only
once all of the following hold:

- Every split reports matching image and label counts in section 8.
- Section 10 reports zero missing labels, zero missing images, zero invalid class IDs, and
  zero structural coordinate errors.
- Section 11 shows polygons that visibly align with the annotated damage.
- The class ordering in the generated `cardd.yaml` matches the ordering displayed in
  section 5.

Until this notebook has been executed on the real dataset, none of these conditions has
been verified.

### Next steps

1. Place the CarDD COCO release under `data/cardd/raw/` and run this notebook end to end.
2. Complete the findings above from the generated tables.
3. Review the class balance in section 10.1 against the imbalance already observed in the
   Carparts-Seg dataset.
4. Check for duplicate or near-duplicate images across the CarDD splits, as was done for
   Carparts-Seg in notebook 03; that check is not part of this conversion notebook.
5. Keep `data/cardd/raw/` untouched, so the conversion can always be reproduced from the
   original release.